# 📄 Document Question Answering System (RAG)

## Overview
This project implements a Retrieval-Augmented Generation (RAG) pipeline that
answers user questions using information retrieved from custom PDF/TXT
documents.

### Tech Stack
- Google Gemini 2.5 Flash
- Pinecone Vector Database
- LangChain
- PyPDF
- Recursive Character Text Splitter

**Developed By:** Udit Gupta

In [33]:


%pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
pypdf \
google-generativeai \
pinecone \
python-dotenv \
tqdm

In [34]:
import os
import re
import uuid
from pathlib import Path
from collections import Counter

from dotenv import load_dotenv

import google.generativeai as genai

from pinecone import Pinecone, ServerlessSpec

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from tqdm import tqdm


# Load Environment Variables
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_HOST = os.getenv("PINECONE_HOST")


# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)


# Local Vector Store (Fallback)
class CustomVectorStore:

    def __init__(self):
        self.records = []

    def upsert(self, vectors):
        for vector in vectors:
            self.records.append(vector)

    def cosine_like(self, a, b):

        # Gemini Embeddings
        if isinstance(a, list) and isinstance(b, list):

            score = 0.0

            for x, y in zip(a, b):
                score += float(x) * float(y)

            return score

        # Local Embeddings
        if isinstance(a, dict) and isinstance(b, dict):

            overlap = set(a.keys()) & set(b.keys())

            return sum(a[token] * b[token] for token in overlap)

        return 0.0

    def query(self, vector, top_k=5, include_metadata=True):

        scores = []

        for vector_id, embedding, metadata in self.records:

            similarity = self.cosine_like(vector, embedding)

            scores.append((similarity, vector_id, metadata))

        scores.sort(key=lambda x: x[0], reverse=True)

        results = []

        for similarity, vector_id, metadata in scores[:top_k]:

            results.append({
                "id": vector_id,
                "score": similarity,
                "metadata": metadata
            })

        return {
            "matches": results
        }


# Backup Embedding
def local_embedding(text):

    words = re.findall(r"[a-zA-Z0-9]+", text.lower())

    return dict(Counter(words))


# Gemini Embedding
def generate_embedding(text):

    try:

        embedding = genai.embed_content(

            model="models/gemini-embedding-001",

            content=text,

            task_type="retrieval_document"

        )

        return embedding["embedding"]

    except Exception:

        return local_embedding(text)


print("Configuration Loaded Successfully")

Configuration Loaded Successfully


In [35]:
pc = None
vector_index = None

if PINECONE_API_KEY:

    try:

        if PINECONE_HOST:

            pc = Pinecone(
                api_key=PINECONE_API_KEY,
                host=PINECONE_HOST
            )

        else:

            pc = Pinecone(
                api_key=PINECONE_API_KEY
            )

        INDEX_NAME = "document-rag"

        existing_indexes = [
            item["name"]
            for item in pc.list_indexes()
        ]

        if INDEX_NAME not in existing_indexes:

            pc.create_index(
                name=INDEX_NAME,
                dimension=768,
                metric="cosine",
                spec=ServerlessSpec(
                    cloud="aws",
                    region="us-east-1"
                )
            )

        vector_index = pc.Index(INDEX_NAME)

        print("Connected to Pinecone Successfully")

    except Exception as e:

        print("Pinecone Error :", e)


if vector_index is None:

    vector_index = CustomVectorStore()

    print("Using Local Vector Store")

Using Local Vector Store


In [36]:
documents = []

DATA_FOLDER = Path(".")

if DATA_FOLDER.exists():

    for file in DATA_FOLDER.iterdir():

        extension = file.suffix.lower()

        if extension == ".pdf":

            loader = PyPDFLoader(str(file))

            documents.extend(loader.load())

        elif extension == ".txt":

            loader = TextLoader(
                str(file),
                encoding="utf-8"
            )

            documents.extend(loader.load())

else:

    print("Create a folder named 'data' and add PDF/TXT files.")

print("-"*50)

print(f"Total Documents Loaded : {len(documents)}")

print("-"*50)

--------------------------------------------------
Total Documents Loaded : 4
--------------------------------------------------


In [37]:
# -----------------------------
# Split Documents
# -----------------------------

splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200

)

document_chunks = splitter.split_documents(documents)

print(f"Total Chunks : {len(document_chunks)}")

# -----------------------------
# Convert into Embeddings
# -----------------------------

vector_records = []

for chunk in tqdm(document_chunks):

    embedding = generate_embedding(
        chunk.page_content
    )

    vector_records.append(

        (

            str(uuid.uuid4()),

            embedding,

            {

                "text": chunk.page_content,

                "source": chunk.metadata.get(
                    "source",
                    "Unknown"
                )

            }

        )

    )

# -----------------------------
# Upload Vectors
# -----------------------------

vector_index.upsert(vector_records)

print()

print("Vector Database Ready")

Total Chunks : 6


100%|██████████| 6/6 [00:00<00:00, 1352.20it/s]


Vector Database Ready


In [38]:
def retrieve_documents(question, top_k=3):

    query_embedding = genai.embed_content(
        model="models/gemini-embedding-001",
        content=question,
        task_type="retrieval_query"
    )["embedding"]

    results = vector_index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    return results["matches"]

In [39]:
model = genai.GenerativeModel("gemini-2.5-flash")


def answer_question(question):

    retrieved_docs = retrieve_documents(question)

    context = ""

    for doc in retrieved_docs:

        context += doc["metadata"]["text"]

        context += "\n\n"

    prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the given context.

If the answer is not available in the document,
reply:

"I couldn't find the answer in the uploaded document."

Context:

{context}

Question:

{question}
"""

    response = model.generate_content(prompt)

    print("="*60)
    print("QUESTION")
    print("="*60)
    print(question)

    print()

    print("="*60)
    print("ANSWER")
    print("="*60)
    print(response.text)

    print()

    print("="*60)
    print("SOURCES")
    print("="*60)

    sources = set()

    for doc in retrieved_docs:

        sources.add(doc["metadata"]["source"])

    for source in sources:

        print("•", source)

In [40]:
import os

print(os.getenv("GEMINI_API_KEY"))

None


In [41]:
import google.generativeai as genai

genai.configure(api_key="YAHAN_APNI_GEMINI_API_KEY_DALO")

In [42]:
model = genai.GenerativeModel("gemini-2.5-flash")

In [43]:
import os

print("Gemini:", os.getenv("GEMINI_API_KEY"))
print("Pinecone:", os.getenv("PINECONE_API_KEY"))

Gemini: None
Pinecone: None


In [44]:
import google.generativeai as genai

GEMINI_API_KEY = "AQ.Ab8RN6LWSZMJIZiAS4X8q4kmT0wnacAGcEKSEJKxMsOy6X8QWw"

genai.configure(api_key=GEMINI_API_KEY)

In [45]:
from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_59r4Ko_UQo3mpgpzAHx8qt2mDAfv9ckT8rjYkNrwio6AJUtBHMeMtUbYUxhJECwSELSvDh")

In [46]:
answer_question(
    "What is Retrieval-Augmented Generation?"
)

QUESTION
What is Retrieval-Augmented Generation?

ANSWER
Retrieval-Augmented Generation (RAG) is a system that answers questions based on custom documents. Instead of relying only on a language model’s internal knowledge, the system retrieves relevant information from documents and then generates answers grounded in that information. This improves factual accuracy and allows question answering over private or domain-specific data. It combines retrieval (finding relevant text chunks), augmentation (adding retrieved content to the model's input), and generation (a language model creating the final answer using the context).

SOURCES
• Week7_Project.pdf


In [47]:
answer_question("What are the objectives of this project?")

QUESTION
What are the objectives of this project?

ANSWER
The objectives of this project are:
*   Understand the concept of Retrieval-Augmented Generation (RAG)
*   Build a pipeline combining retrieval and generation
*   Enable question answering over custom documents such as PDFs or text files
*   Learn how modern AI systems work internally

SOURCES
• Week7_Project.pdf


In [48]:
answer_question("Explain the workflow of RAG.")

QUESTION
Explain the workflow of RAG.

ANSWER
The workflow of RAG consists of the following stages:

1.  **Document Ingestion**: Documents like PDFs or text files are loaded and converted into raw text.
2.  **Text Chunking**: The text is split into smaller chunks to improve retrieval accuracy.
3.  **Embedding Creation**: Each chunk is converted into a vector representation capturing its semantic meaning.
4.  **Vector Database**: Embeddings are stored in a vector database for efficient similarity search.
5.  **Query Processing**: The user's question is converted into an embedding.
6.  **Context Retrieval**: The system retrieves the most relevant chunks from the database.
7.  **Answer Generation**: A language model generates the final answer using the retrieved context.

SOURCES
• Week7_Project.pdf
